# Kaggle Predict F1 Pit Stops: LightGBM + XGBoost Ensemble

This notebook tests a simple probability average between tuned LightGBM and XGBoost.

The goal is to check whether model diversity improves OOF validation before using the ensemble for `submission.csv`.

## 1. Setup

In [ ]:
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.labelsize": 10, "legend.frameon": False})

RANDOM_STATE = 42
TARGET = "PitNextLap"
ID_COL = "id"

# Use True for quick checks, then False for final ensemble evidence.
RUN_FAST = True
FAST_SAMPLE_SIZE = 180_000
N_SPLITS = 5

# If feature validation selects a different feature set, update this value.
SELECTED_FEATURE_SET = "safe_plus_ratios"

## 2. Load Data

In [ ]:
def find_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e5"),
        Path("/kaggle/input/playground-series-s6e5"),
        Path("../input/competitions/playground-series-s6e5"),
        Path("../input/playground-series-s6e5"),
        Path("data"),
        Path("../data"),
        Path("."),
    ]
    for path in candidates:
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError("Could not find train.csv and test.csv. Update DATA_DIR manually.")


def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        dtype = out[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="integer")
        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")
        elif pd.api.types.is_object_dtype(dtype):
            nunique = out[col].nunique(dropna=False)
            if nunique / max(len(out), 1) < 0.5:
                out[col] = out[col].astype("category")
    return out


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = reduce_memory_usage(pd.read_csv(DATA_DIR / "train.csv"))
test = reduce_memory_usage(pd.read_csv(DATA_DIR / "test.csv"))
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if RUN_FAST and len(train) > FAST_SAMPLE_SIZE:
    train_eval = (
        train.groupby(TARGET, group_keys=False)
        .apply(lambda x: x.sample(frac=min(1.0, FAST_SAMPLE_SIZE / len(train)), random_state=RANDOM_STATE))
        .sample(frac=1.0, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )
else:
    train_eval = train.copy()

print(f"DATA_DIR: {DATA_DIR}")
print(f"train_eval: {train_eval.shape}")
print(f"test: {test.shape}")
print(f"target positive rate: {train_eval[TARGET].mean():.5f}")

## 3. Feature Engineering

In [ ]:
feature_set_configs = {
    "raw": {"include_safe": False, "include_ratios": False, "drop_cols": []},
    "safe_engineered": {"include_safe": True, "include_ratios": False, "drop_cols": []},
    "safe_plus_ratios": {"include_safe": True, "include_ratios": True, "drop_cols": []},
    "safe_plus_ratios_no_driver": {"include_safe": True, "include_ratios": True, "drop_cols": ["Driver"]},
    "safe_plus_ratios_no_pitstop": {"include_safe": True, "include_ratios": True, "drop_cols": ["PitStop"]},
}


def add_features(df: pd.DataFrame, include_safe: bool = True, include_ratios: bool = True) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6

    if include_safe and {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["EstimatedRaceLaps"] = out["LapNumber"] / out["RaceProgress"].clip(lower=eps)
        out["EstimatedLapsRemaining"] = out["EstimatedRaceLaps"] - out["LapNumber"]
        out["LapNumber_x_RaceProgress"] = out["LapNumber"] * out["RaceProgress"]

    if include_ratios and {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["TyreLife_to_LapNumber"] = out["TyreLife"] / out["LapNumber"].clip(lower=eps)

    if include_ratios and {"TyreLife", "EstimatedRaceLaps"}.issubset(out.columns):
        out["TyreLife_to_EstimatedRaceLaps"] = out["TyreLife"] / out["EstimatedRaceLaps"].clip(lower=eps)

    if include_safe and {"LapTime (s)", "LapTime_Delta"}.issubset(out.columns):
        out["LapTime_plus_Delta"] = out["LapTime (s)"] + out["LapTime_Delta"]
        out["AbsLapTime_Delta"] = out["LapTime_Delta"].abs()

    if include_safe and {"Position", "Position_Change"}.issubset(out.columns):
        out["PreviousPositionApprox"] = out["Position"] - out["Position_Change"]
        out["AbsPosition_Change"] = out["Position_Change"].abs()

    if include_safe and "Compound" in out.columns:
        compound = out["Compound"].astype(str)
        out["IsSoft"] = (compound == "SOFT").astype("int8")
        out["IsMedium"] = (compound == "MEDIUM").astype("int8")
        out["IsHard"] = (compound == "HARD").astype("int8")
        out["IsWetOrIntermediate"] = compound.isin(["WET", "INTERMEDIATE"]).astype("int8")

    return reduce_memory_usage(out)


def prepare_feature_set(df: pd.DataFrame, feature_set: str) -> pd.DataFrame:
    config = feature_set_configs[feature_set]
    out = add_features(df, include_safe=config["include_safe"], include_ratios=config["include_ratios"])
    return out.drop(columns=[c for c in config.get("drop_cols", []) if c in out.columns])


train_model = prepare_feature_set(train_eval, SELECTED_FEATURE_SET)
test_model = prepare_feature_set(test, SELECTED_FEATURE_SET)

X = train_model.drop(columns=[TARGET])
y = train_model[TARGET].astype("int8")
X_test = test_model.copy()

print("Feature set:", SELECTED_FEATURE_SET)
print("Features:", X.shape[1])

## 4. Preprocessing and Models

In [ ]:
def get_feature_columns(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    features = df.drop(columns=[c for c in [TARGET, ID_COL] if c in df.columns])
    cat_cols = features.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in features.columns if c not in cat_cols]
    return num_cols, cat_cols


def make_preprocessor(df: pd.DataFrame) -> ColumnTransformer:
    num_cols, cat_cols = get_feature_columns(df)
    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ], remainder="drop")


def evaluate_predictions(y_true, y_pred) -> dict:
    y_pred = np.clip(y_pred, 1e-6, 1 - 1e-6)
    return {
        "roc_auc": roc_auc_score(y_true, y_pred),
        "average_precision": average_precision_score(y_true, y_pred),
        "log_loss": log_loss(y_true, y_pred),
    }


lgbm_params = {
    "objective": "binary",
    "n_estimators": 900 if RUN_FAST else 1800,
    "learning_rate": 0.02,
    "num_leaves": 127,
    "min_child_samples": 150,
    "subsample": 0.95,
    "colsample_bytree": 0.9,
    "reg_alpha": 1.0,
    "reg_lambda": 10.0,
    "max_bin": 255,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": -1,
}

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "n_estimators": 700 if RUN_FAST else 1400,
    "learning_rate": 0.025,
    "max_depth": 7,
    "min_child_weight": 8,
    "subsample": 0.85,
    "colsample_bytree": 0.9,
    "reg_alpha": 0.1,
    "reg_lambda": 3.0,
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

## 5. Cross-Validate Individual Models

In [ ]:
def cross_validate_model(model_name: str, model_factory) -> tuple[np.ndarray, list[dict]]:
    oof = np.zeros(len(X), dtype=np.float32)
    rows = []
    start = time.time()

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        X_train_raw, X_valid_raw = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        preprocessor = make_preprocessor(X_train_raw)
        X_train = preprocessor.fit_transform(X_train_raw)
        X_valid = preprocessor.transform(X_valid_raw)

        model = model_factory()
        model.fit(X_train, y_train)
        pred = model.predict_proba(X_valid)[:, 1]
        oof[valid_idx] = pred

        scores = evaluate_predictions(y_valid, pred)
        scores.update({"model": model_name, "fold": fold})
        rows.append(scores)
        print(f"{model_name} fold {fold}: auc={scores['roc_auc']:.5f}, logloss={scores['log_loss']:.5f}")

        del preprocessor, X_train, X_valid, model
        gc.collect()

    overall = evaluate_predictions(y, oof)
    overall.update({"model": model_name, "fold": "oof", "fit_seconds": time.time() - start})
    rows.append(overall)
    print(f"{model_name} OOF: auc={overall['roc_auc']:.5f}, ap={overall['average_precision']:.5f}, logloss={overall['log_loss']:.5f}")
    return oof, rows


lgbm_oof, lgbm_rows = cross_validate_model("lightgbm", lambda: LGBMClassifier(**lgbm_params))
xgb_oof, xgb_rows = cross_validate_model("xgboost", lambda: XGBClassifier(**xgb_params))

model_results = pd.DataFrame(lgbm_rows + xgb_rows)
model_results[model_results["fold"].eq("oof")]

## 6. Tune Blend Weight

In [ ]:
blend_rows = []
for lgbm_weight in np.linspace(0, 1, 21):
    blend_pred = lgbm_weight * lgbm_oof + (1 - lgbm_weight) * xgb_oof
    scores = evaluate_predictions(y, blend_pred)
    scores.update({
        "model": "lgbm_xgb_blend",
        "lgbm_weight": lgbm_weight,
        "xgb_weight": 1 - lgbm_weight,
    })
    blend_rows.append(scores)

blend_results = pd.DataFrame(blend_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
blend_results.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in zip(axes, ["roc_auc", "average_precision", "log_loss"], ["ROC AUC", "Average Precision", "Log Loss"]):
    sns.lineplot(data=blend_results.sort_values("lgbm_weight"), x="lgbm_weight", y=metric, marker="o", color=sns.color_palette("viridis", 8)[4], ax=ax)
    ax.set_title(title)
    ax.set_xlabel("LightGBM weight")
plt.tight_layout()
plt.show()

## 7. Train Final Ensemble and Submit

In [ ]:
best_blend = blend_results.iloc[0]
best_lgbm_weight = float(best_blend["lgbm_weight"])
best_xgb_weight = float(best_blend["xgb_weight"])

print("Best blend:")
print(best_blend)

final_preprocessor = make_preprocessor(X)
X_train_processed = final_preprocessor.fit_transform(X)
X_test_processed = final_preprocessor.transform(X_test)

final_lgbm = LGBMClassifier(**lgbm_params)
final_xgb = XGBClassifier(**xgb_params)

final_lgbm.fit(X_train_processed, y)
final_xgb.fit(X_train_processed, y)

lgbm_test_pred = final_lgbm.predict_proba(X_test_processed)[:, 1]
xgb_test_pred = final_xgb.predict_proba(X_test_processed)[:, 1]
ensemble_pred = best_lgbm_weight * lgbm_test_pred + best_xgb_weight * xgb_test_pred
ensemble_pred = np.clip(ensemble_pred, 1e-6, 1 - 1e-6)

submission = sample_submission.copy()
submission[TARGET] = ensemble_pred
submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

model_results.to_csv(OUTPUT_DIR / "ensemble_model_results.csv", index=False)
blend_results.to_csv(OUTPUT_DIR / "ensemble_blend_results.csv", index=False)

print("Saved:", submission_path)
submission.head()

## 8. Decision Rule

Use the ensemble only if its OOF score beats the best individual model by a meaningful margin or improves log loss without hurting AUC. If the best weight is close to `1.0`, the ensemble is not adding enough over tuned LightGBM.